# Build Mini Clef From Scratch: Qwen, LoRA and a Decision Head

**What you'll learn**
- Clef's idea: a decoder reads the state and the question once; a small head scores every option
- Put the state first, so every option token has already read it
- LoRA on the decoder plus a new decision head, trained together by `Trainer`
- Calibrate, save, load back, answer a System One request
- Lab: every model in the course on the card problem

## Setup
- The typed questions saved by the Mini Laya section, in `laya`'s JSONL format: same train, calibration, test and transfer rows

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, load_dataset
from safetensors.torch import load_file, save_file
from transformers import AutoModel, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, PeftModel, get_peft_model
import laya
from laya import evals
from laya.common import collate_items, render_options, serialize_state, QTYPES
from laya.train import read_jsonl, to_internal, target_from_expected, evaluate_records
from laya.calibrate import fit_temperature_map

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token

In [ ]:
data = {}
for name in ["train", "calibration", "test", "transfer"]:
    data[name] = read_jsonl(f"runs/system-one-data/{name}.jsonl")  # the rows saved by the Mini Laya section
    print(name, len(data[name]))

In [ ]:
data["train"][0]

## Prepare the Model Input: State, Then Question and Options
- `to_internal`, `target_from_expected`: the question and its answer exactly as Laya reads them
- A decoder reads left to right: with the state first, every option token has seen it
- One answer slot per option: the hidden state at the option's last token
- Training rows shuffle their choice options, so a position never means an answer
- First the steps on one row, then the same steps as a function

In [ ]:
row = data["train"][0]
qid = list(row["questions"])[0]  # every row asks one question
qid

In [ ]:
q = to_internal(qid, row["questions"][qid])  # the question exactly as Laya reads it
q

In [ ]:
options = render_options(q)  # one line of text per option
options

In [ ]:
rng = np.random.default_rng(0)  # shuffles the options of training rows
order = list(rng.permutation(len(options)))  # a new order, so a position never means an answer
order

In [ ]:
def ids(text, n=None):
    return tok(text, add_special_tokens=False).input_ids[:n]  # token ids, at most n of them

ids("\nOptions:")

In [ ]:
state = serialize_state(row["state"])
seq = ids(f"STATE:\n{state}", 384)  # the state first
seq += ids(f"\n\n{q['t']} question: {q['ins']}", 96)  # then the question
seq += ids("\nOptions:")
print(tok.decode(seq))

In [ ]:
markers = []
for i in order:
    seq += ids(f"\n- {options[i]}", 48)  # one option
    markers.append(len(seq) - 1)  # its answer slot: the option's last token

markers

In [ ]:
target = target_from_expected(q, row["expected"][qid])  # one number per option, 1 for the right one
target

In [ ]:
shuffled_target = []
for i in order:
    shuffled_target.append(target[i])  # in the same order as the options
shuffled_target

In [ ]:
def encode(row, shuffle=False):
    qid = list(row["questions"])[0]
    q = to_internal(qid, row["questions"][qid])
    options = render_options(q)

    order = list(range(len(options)))
    if shuffle and q["t"] == "choice":
        order = list(rng.permutation(len(options)))

    state = serialize_state(row["state"])
    seq = ids(f"STATE:\n{state}", 384)
    seq += ids(f"\n\n{q['t']} question: {q['ins']}", 96)
    seq += ids("\nOptions:")

    markers = []
    for i in order:
        seq += ids(f"\n- {options[i]}", 48)
        markers.append(len(seq) - 1)

    x = {"ids": seq, "markers": markers, "qtype": QTYPES[q["t"]]}
    if "expected" in row:  # labelled rows also get their target
        target = target_from_expected(q, row["expected"][qid])
        x["target"] = []
        for i in order:
            x["target"].append(target[i])
    return x

In [ ]:
x = encode(data["train"][0])
print(tok.decode(x["ids"]))

In [ ]:
for slot, value in zip(x["markers"], x["target"]):
    print(repr(tok.decode(x["ids"][slot])), value)  # the token at each answer slot, and its target

In [ ]:
def to_dataset(rows, shuffle=False):
    encoded = []
    for row in rows:
        encoded.append(encode(row, shuffle))
    return Dataset.from_list(encoded)

In [ ]:
train_ds = to_dataset(data["train"], shuffle=True)
cal_ds = to_dataset(data["calibration"])
test_ds = to_dataset(data["test"])
transfer_ds = to_dataset(data["transfer"])
train_ds

## Mini Clef's Architecture: Qwen, LoRA and a Decision Head
- `AutoModel`: Qwen without its word head, it returns hidden states
- LoRA adapters on every linear layer of the decoder
- Decision head: one attention layer lets the options compare notes, then one logit per option
- `forward` returns the loss, cross-entropy against the target, so `Trainer` can train it

In [ ]:
class MiniClef(torch.nn.Module):
    def __init__(self, backbone):
        super().__init__()
        d = backbone.config.hidden_size  # numbers per token
        self.backbone = backbone
        self.type_emb = torch.nn.Embedding(3, d)  # one vector per question type
        self.compare = torch.nn.TransformerEncoderLayer(d, 14, 2 * d, batch_first=True, norm_first=True)
        self.score = torch.nn.Sequential(torch.nn.LayerNorm(d), torch.nn.Linear(d, 1))

    def forward(self, input_ids, attention_mask, marker_pos, marker_mask, qtype, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state.float()  # one vector per token
        index = marker_pos[..., None].expand(-1, -1, h.size(-1))
        slots = h.gather(1, index)  # the vectors at the answer slots
        slots = slots + self.type_emb(qtype)[:, None]  # plus the question type
        slots = self.compare(slots, src_key_padding_mask=~marker_mask)  # the options compare notes
        logits = self.score(slots).squeeze(-1)  # one logit per option
        logits = logits.masked_fill(~marker_mask, -1e4)  # padding slots never win
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels)
        return {"loss": loss, "logits": logits}

In [ ]:
LORA = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear")
LORA

In [ ]:
torch.manual_seed(0)
backbone = AutoModel.from_pretrained(MODEL, dtype=torch.bfloat16)  # Qwen without its word head
backbone.config.hidden_size

In [ ]:
backbone = get_peft_model(backbone, LORA)  # plus LoRA adapters
backbone.print_trainable_parameters()

In [ ]:
model = MiniClef(backbone)  # plus the new decision head
model.score

In [ ]:
total = 0
trainable = 0
head = 0
for name, p in model.named_parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()  # LoRA adapters and the head
    if not name.startswith("backbone."):
        head += p.numel()  # the decision head alone

total, trainable, head

## Set Up the Hugging Face Trainer for Mini Clef
- `collate_items` from `laya` pads ids and answer slots; Qwen's pad token fills the gaps
- `records`: `trainer.predict`, cut to each row's own options, in the format `laya`'s scoring tools read

In [ ]:
COLUMNS = ["input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype"]

def collate(batch):
    b = collate_items([batch], tok.pad_token_id)  # pads ids and answer slots
    inputs = {}
    for k in COLUMNS:
        inputs[k] = b[k]
    inputs["labels"] = b.get("target")  # None for rows without a target
    return inputs

In [ ]:
batch = collate([train_ds[0], train_ds[1]])  # two rows, padded to the same length
for k, v in batch.items():
    print(k, v.shape)

In [ ]:
def make_trainer(ds, lr=2e-4, epochs=1):
    args = TrainingArguments(
        "runs/mini-clef-trainer",
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,  # 4 rows at a time, 16 rows per update
        per_device_eval_batch_size=32,
        warmup_steps=0.06,
        bf16=True,
        logging_steps=200,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,  # collate needs ids, markers and qtype
        seed=0,
    )
    return Trainer(model=model, args=args, train_dataset=ds, data_collator=collate)

In [ ]:
def records(trainer, ds):
    out = trainer.predict(ds)  # logits and targets for every row, padded to the widest row
    rows = []
    for qtype, markers, logits, target in zip(ds["qtype"], ds["markers"], out.predictions, out.label_ids):
        k = len(markers)  # this row's own number of options
        rows.append((qtype, logits[:k], target[:k], k))
    return rows

## Baseline Accuracy Before Training
- LoRA starts as a no-op and the head is random: near chance

In [ ]:
trainer = make_trainer(train_ds)
test_records = records(trainer, test_ds)
test_records[0]

In [ ]:
evaluate_records(test_records)

## Train Mini Clef: LoRA and the Decision Head Together
- One epoch over every training row, learning rate `2e-4`, bf16

In [ ]:
torch.cuda.reset_peak_memory_stats()
trainer.train()
torch.cuda.max_memory_allocated() / 1024**3

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
log[["epoch", "loss"]].dropna()

## Calibrate With One Temperature per Question Type
- The same `fit_temperature_map` as Mini Laya, on the calibration rows

In [ ]:
cal_records = records(trainer, cal_ds)
fit = fit_temperature_map(cal_records)
fit.keys()

In [ ]:
T = fit["temperature"]  # one temperature per question type
T_opt = fit["temperature_by_options"]  # and per number of options
T, T_opt

In [ ]:
test_records = records(trainer, test_ds)
transfer_records = records(trainer, transfer_ds)
len(test_records), len(transfer_records)

In [ ]:
results = {
    "test, T=1": evaluate_records(test_records),
    "test, fitted T": evaluate_records(test_records, T, T_opt),
    "transfer, fitted T": evaluate_records(transfer_records, T, T_opt),
}
table = pd.DataFrame.from_dict(results, orient="index")
table.drop(columns="by_type")

## Save Mini Clef and Load It Back
- The LoRA adapter with `save_pretrained`: only the adapter, the base model downloads separately
- The head and the temperatures next to it

In [ ]:
OUT = "runs/mini-clef"
model.backbone.save_pretrained(OUT)  # the LoRA adapter only
os.listdir(OUT)

In [ ]:
head_weights = {}
for name, weight in model.state_dict().items():
    if not name.startswith("backbone."):
        head_weights[name] = weight  # the decision head

list(head_weights)

In [ ]:
save_file(head_weights, f"{OUT}/head.safetensors")

settings = {"base": MODEL, "temperature": T}
with open(f"{OUT}/mini_clef.json", "w") as f:
    json.dump(settings, f)
settings

In [ ]:
for name in os.listdir(OUT):
    path = f"{OUT}/{name}"
    if os.path.isfile(path):
        print(name, round(os.path.getsize(path) / 1024**2, 1), "MB")

In [ ]:
del trainer, model
torch.cuda.empty_cache()

base = AutoModel.from_pretrained(MODEL, dtype=torch.bfloat16)
backbone = PeftModel.from_pretrained(base, OUT, is_trainable=True)  # trainable: the Lab trains it further
backbone.print_trainable_parameters()

In [ ]:
model = MiniClef(backbone).cuda()
head_weights = load_file(f"{OUT}/head.safetensors")
model.load_state_dict(head_weights, strict=False)  # strict=False: the backbone is already loaded
model.eval();

## Run Mini Clef on a System One Request
- Same request shape as the Ollama models and Laya: a state plus typed questions
- First the steps for one question, then the same steps as a function

In [ ]:
QUESTIONS = {
    "sentiment": {
        "type": "score",
        "instructions": "How does the customer feel?",
        "criteria": ["very negative", "negative", "neutral", "positive", "very positive"],
    },
    "topic": {
        "type": "choice",
        "instructions": "What is it about?",
        "criteria": ["travel", "banking", "food"],
    },
    "refund": {
        "type": "noul",
        "instructions": "Is the customer likely to ask for a refund?",
    },
}
STATE = "My flight was cancelled and nobody at the desk would help."

In [ ]:
x = encode({"state": STATE, "questions": {"topic": QUESTIONS["topic"]}})  # no expected, so no target
batch = collate([x])
inputs = {}
for k in COLUMNS:
    inputs[k] = batch[k].cuda()
inputs["marker_pos"]

In [ ]:
with torch.no_grad():
    logits = model(**inputs)["logits"][0]  # one logit per option
logits

In [ ]:
t = T[QTYPES["choice"]]  # the choice questions' temperature
probs = torch.softmax(logits / t, -1)
probs

In [ ]:
@torch.no_grad()
def system_one(state, questions):
    answers = {}
    for qid, question in questions.items():
        q = to_internal(qid, question)
        x = encode({"state": state, "questions": {qid: question}})
        batch = collate([x])
        inputs = {}
        for k in COLUMNS:
            inputs[k] = batch[k].cuda()
        logits = model(**inputs)["logits"][0]
        t = T[QTYPES[q["t"]]]  # this question type's temperature
        probs = torch.softmax(logits / t, -1)
        answers[qid] = dict(zip(render_options(q), probs.tolist()))
    return answers

In [ ]:
system_one(STATE, QUESTIONS)

## Lab: Laya vs Mini Laya vs Mini Clef on the Card Task
- The ten card intents and the same splits as the fundamentals sections
- Mini Clef: fine-tune it further on the card messages, the same `make_trainer`
- Laya, Mini Laya and their card versions: the checkpoints saved in earlier sections, scored with `laya.evals`

In [ ]:
INTENTS = {
    "card_arrival": "ordered card has not arrived yet",
    "card_delivery_estimate": "asks how long card delivery takes",
    "activate_my_card": "how to activate a card",
    "card_not_working": "card stopped working or is not accepted",
    "lost_or_stolen_card": "card was lost or stolen",
    "compromised_card": "card details may be known to someone else",
    "card_payment_not_recognised": "a card payment the customer did not make",
    "declined_card_payment": "a card payment was declined",
    "card_swallowed": "an ATM kept the card",
    "contactless_not_working": "tap to pay does not work",
}

QUESTION = {
    "type": "choice",
    "instructions": "What does the customer need help with?",
    "criteria": INTENTS,
}

In [ ]:
banking = load_dataset("mteb/banking77")

def is_card_intent(row):
    return row["label_text"] in INTENTS  # one of our ten card intents?

cards = banking.filter(is_card_intent)
split = cards["train"].train_test_split(test_size=200, seed=0)  # the same splits as the fundamentals sections
split

In [ ]:
def to_rows(messages):
    rows = []
    for message in messages:
        rows.append({
            "state": message["text"],
            "questions": {"intent": QUESTION},
            "expected": {"intent": message["label_text"]},
        })
    return rows

train_rows = to_rows(split["train"])
val_rows = to_rows(split["test"])
test_rows = to_rows(cards["test"])
len(train_rows), len(val_rows), len(test_rows)

In [ ]:
card_ds = {
    "train": to_dataset(train_rows, shuffle=True),
    "val": to_dataset(val_rows),
    "test": to_dataset(test_rows),
}
card_ds["train"]

In [ ]:
def count_parameters(m):
    total = 0
    for p in m.parameters():
        total += p.numel()
    return total / 1e6  # in millions

count_parameters(model)

In [ ]:
results = {}
cards_trainer = make_trainer(card_ds["train"], lr=1e-4, epochs=2)

test_records = records(cards_trainer, card_ds["test"])
before = evaluate_records(test_records, T)  # Mini Clef as saved, with its own temperatures
before["parameters (M)"] = count_parameters(model)
results["Mini Clef"] = before
before["accuracy"]

In [ ]:
cards_trainer.train()

In [ ]:
val_records = records(cards_trainer, card_ds["val"])
card_T = fit_temperature_map(val_records)["temperature"]  # new temperatures, fitted on the validation messages
card_T

In [ ]:
test_records = records(cards_trainer, card_ds["test"])
after = evaluate_records(test_records, card_T)
after["parameters (M)"] = count_parameters(model)
results["Mini Clef + cards"] = after
after["accuracy"]

In [ ]:
table = pd.DataFrame.from_dict(results, orient="index")
table.drop(columns="by_type")

In [ ]:
card_test = evals.Dataset.from_jsonl("runs/cards/test.jsonl")

CHECKPOINTS = {
    "Laya": "convaiinnovations/laya",
    "Laya + cards": "runs/laya-cards",
    "Mini Laya": "runs/mini-laya",
    "Mini Laya + cards": "runs/mini-laya-cards",
}

for name, path in CHECKPOINTS.items():
    agent = laya.Agent(path, device="cuda")
    report = evals.evaluate(agent, card_test)
    results[name] = {
        "accuracy": report.overall["choice_accuracy"],
        "ece": report.overall["ece"],
        "parameters (M)": count_parameters(agent.model),
    }
    print(name, results[name]["accuracy"])

In [ ]:
table = pd.DataFrame.from_dict(results, orient="index")
table[["accuracy", "ece", "parameters (M)"]].round(3)

## Cheat Sheet
| Step | Code |
|---|---|
| Data | `read_jsonl("runs/system-one-data/train.jsonl")`, saved by the Mini Laya section |
| Question and answer | `to_internal`, `target_from_expected` from `laya.train` |
| Input | state first, then question and options; one answer slot per option |
| Backbone | `get_peft_model(AutoModel.from_pretrained(MODEL), LoraConfig(target_modules="all-linear"))` |
| Head | type embedding, one attention layer across options, one logit per option |
| Loss | cross-entropy against the target, returned by `forward` |
| Train | `Trainer`, `learning_rate=2e-4`, 1 epoch, `bf16=True` |
| Score | `trainer.predict` as records, then `evaluate_records`: accuracy, ECE |
| Calibration | `fit_temperature_map` on calibration rows |
| Ship | adapter with `save_pretrained`, `head.safetensors`, temperatures in `mini_clef.json` |
| Load | `PeftModel.from_pretrained(base, path)`, then the head's state dict |